# 面试问题：Agent OAuth 委托授权怎样防止 Confused Deputy？

可直接复述的回答：Agent 不应持有用户长期凭据，而应通过 OAuth 获得短期、精确 audience 和 scope 的委托 token。Scope 只说明动作类型，不等于对象级授权；资源服务器还要检查 tenant、订单归属和主体。Token exchange 必须限制目标 audience，避免把 A 服务 token 拿到 B 服务使用。高风险动作需要 step-up approval，并把动作参数绑定到票据。DPoP 或 mTLS 可把 token 绑定客户端密钥，jti/nonce 防重放。所有验证在资源服务器完成，不能相信 Agent 自报权限。失败应返回稳定原因码并写入审计账本。

后续实验使用可读的小型业务数据验证关键判断。所有数值都标记为教学实验，不代表真实 GPU、线上流量或基础模型泛化结果。


## 1. 真实案例：订单 Agent 委托 Token 与输入预览

六条脱敏请求包含 subject、tenant、audience、scope、资源、DPoP proof、jti、金额和审批。案例覆盖正常查询、错误 audience、跨租户、重放和大额退款。


In [1]:
requests11 = [  # 构造六条委托授权请求。
    {"id": "r1", "sub": "u17", "tenant": "shop-a", "aud": "orders-api", "scope": {"orders.read"}, "resource": "o-1", "action": "read", "amount": 0, "proof": "key-u17", "jti": "j1", "approval": None},  # 正常订单查询。
    {"id": "r2", "sub": "u17", "tenant": "shop-a", "aud": "profile-api", "scope": {"orders.read"}, "resource": "o-1", "action": "read", "amount": 0, "proof": "key-u17", "jti": "j2", "approval": None},  # 错误 audience 的 confused deputy。
    {"id": "r3", "sub": "u17", "tenant": "shop-b", "aud": "orders-api", "scope": {"orders.read"}, "resource": "o-1", "action": "read", "amount": 0, "proof": "key-u17", "jti": "j3", "approval": None},  # 跨租户资源访问。
    {"id": "r4", "sub": "u17", "tenant": "shop-a", "aud": "orders-api", "scope": {"orders.refund"}, "resource": "o-2", "action": "refund", "amount": 80, "proof": "key-u17", "jti": "j4", "approval": None},  # 正常小额退款。
    {"id": "r5", "sub": "u17", "tenant": "shop-a", "aud": "orders-api", "scope": {"orders.refund"}, "resource": "o-3", "action": "refund", "amount": 1200, "proof": "key-u17", "jti": "j5", "approval": None},  # 大额退款缺 step-up。
    {"id": "r6", "sub": "u17", "tenant": "shop-a", "aud": "orders-api", "scope": {"orders.read"}, "resource": "o-1", "action": "read", "amount": 0, "proof": "key-u17", "jti": "j1", "approval": None},  # 重放第一个 jti。
]  # 完成授权成功与失败案例。
resource_acl11 = {"o-1": ("u17", "shop-a"), "o-2": ("u17", "shop-a"), "o-3": ("u17", "shop-a")}  # 定义资源服务器权威对象归属。
print("教学实验输入：OAuth委托请求")  # 输出请求预览表头。
for request11 in requests11:  # 逐条展示 token claims 与动作。
    print(request11)  # 输出一条授权请求。


教学实验输入：OAuth委托请求
{'id': 'r1', 'sub': 'u17', 'tenant': 'shop-a', 'aud': 'orders-api', 'scope': {'orders.read'}, 'resource': 'o-1', 'action': 'read', 'amount': 0, 'proof': 'key-u17', 'jti': 'j1', 'approval': None}
{'id': 'r2', 'sub': 'u17', 'tenant': 'shop-a', 'aud': 'profile-api', 'scope': {'orders.read'}, 'resource': 'o-1', 'action': 'read', 'amount': 0, 'proof': 'key-u17', 'jti': 'j2', 'approval': None}
{'id': 'r3', 'sub': 'u17', 'tenant': 'shop-b', 'aud': 'orders-api', 'scope': {'orders.read'}, 'resource': 'o-1', 'action': 'read', 'amount': 0, 'proof': 'key-u17', 'jti': 'j3', 'approval': None}
{'id': 'r4', 'sub': 'u17', 'tenant': 'shop-a', 'aud': 'orders-api', 'scope': {'orders.refund'}, 'resource': 'o-2', 'action': 'refund', 'amount': 80, 'proof': 'key-u17', 'jti': 'j4', 'approval': None}
{'id': 'r5', 'sub': 'u17', 'tenant': 'shop-a', 'aud': 'orders-api', 'scope': {'orders.refund'}, 'resource': 'o-3', 'action': 'refund', 'amount': 1200, 'proof': 'key-u17', 'jti': 'j5', 'approval': N

## 2. Baseline（基线）：只检查 Scope

朴素资源服务器只看 `orders.read/refund`，会放过错误 audience 和跨租户请求，也没有 proof 和 replay 防护。


In [2]:
def scope_only11(request11):  # 实现只检查动作 scope 的错误基线。
    required11 = "orders.read" if request11["action"] == "read" else "orders.refund"  # 根据动作选择所需 scope。
    return required11 in request11["scope"]  # 忽略 audience、对象和 proof。
baseline_rows11 = [(request11["id"], scope_only11(request11)) for request11 in requests11]  # 运行全部 scope-only 决策。
print("Scope-only基线：request | allowed")  # 输出基线授权表头。
for row11 in baseline_rows11:  # 逐请求展示错误放行。
    print(row11)  # 输出一条 scope 检查结果。


Scope-only基线：request | allowed
('r1', True)
('r2', True)
('r3', True)
('r4', True)
('r5', True)
('r6', True)


## 3. 核心实现：Audience、ACL、DPoP、Replay 与 Step-up

资源服务器按固定顺序校验 audience、scope、服务端 ACL、proof、jti 和高金额审批。jti 只在前置验证通过后登记，避免失败请求污染 replay cache。


In [3]:
seen_jti11 = set()  # 保存已经成功验证的 DPoP jti。
audit11 = []  # 收集授权决策账本。
def authorize11(request11):  # 实现资源服务器端委托授权。
    if request11["aud"] != "orders-api":  # 检查 token 是否发给当前资源服务器。
        return False, "audience_mismatch"  # 阻止跨服务 token 被误用。
    required11 = "orders.read" if request11["action"] == "read" else "orders.refund"  # 计算动作所需 scope。
    if required11 not in request11["scope"]:  # 检查委托动作范围。
        return False, "scope_missing"  # 拒绝 scope 不足。
    if resource_acl11.get(request11["resource"]) != (request11["sub"], request11["tenant"]):  # 使用权威 ACL 检查对象归属和租户。
        return False, "resource_forbidden"  # 阻止跨用户或跨租户访问。
    if request11["proof"] != f"key-{request11['sub']}":  # 检查 token 是否绑定正确客户端密钥。
        return False, "proof_mismatch"  # 拒绝被盗 token 的错误 proof。
    if request11["jti"] in seen_jti11:  # 检查一次性 proof 标识是否重放。
        return False, "replay_detected"  # 拒绝重复请求。
    if request11["action"] == "refund" and request11["amount"] > 500 and request11["approval"] != f"approve:{request11['resource']}:{request11['amount']}":  # 校验大额动作绑定审批。
        return False, "step_up_required"  # 阻止无审批或参数不匹配的退款。
    seen_jti11.add(request11["jti"])  # 在全部检查通过后登记防重放标识。
    return True, "allowed"  # 允许最小权限动作。
for request11 in requests11:  # 按到达顺序执行授权。
    allowed11, reason11 = authorize11(request11)  # 获取独立策略裁决。
    audit11.append((request11["id"], allowed11, reason11))  # 保存稳定原因码。
print("授权审计：request | allowed | reason")  # 输出核心授权账本表头。
for row11 in audit11:  # 逐条展示每层门禁结果。
    print(row11)  # 输出一条资源服务器决策。


授权审计：request | allowed | reason
('r1', True, 'allowed')
('r2', False, 'audience_mismatch')
('r3', False, 'resource_forbidden')
('r4', True, 'allowed')
('r5', False, 'step_up_required')
('r6', False, 'replay_detected')


## 4. 结果表与结果解读

Scope-only 放行六条请求；严格授权只允许正常查询和小额退款。错误 audience、跨租户、大额无审批和 jti 重放分别得到稳定拒绝原因。


In [4]:
baseline_allowed11 = sum(row11[1] for row11 in baseline_rows11)  # 统计 scope-only 放行数。
core_allowed11 = sum(row11[1] for row11 in audit11)  # 统计严格授权放行数。
print("方法 | allowed | wrong-aud通过 | cross-tenant通过 | replay通过")  # 输出授权方案对照表头。
print("scope_only", baseline_allowed11, dict(baseline_rows11)["r2"], dict(baseline_rows11)["r3"], dict(baseline_rows11)["r6"])  # 展示 confused deputy 风险。
print("delegated_policy", core_allowed11, dict((row11[0], row11[1]) for row11 in audit11)["r2"], dict((row11[0], row11[1]) for row11 in audit11)["r3"], dict((row11[0], row11[1]) for row11 in audit11)["r6"])  # 展示严格资源授权。
print("结果解读：scope是动作上限，audience和对象ACL决定token能在哪里作用于什么资源")  # 解释三种授权维度。


方法 | allowed | wrong-aud通过 | cross-tenant通过 | replay通过
scope_only 6 True True True
delegated_policy 2 False False False
结果解读：scope是动作上限，audience和对象ACL决定token能在哪里作用于什么资源


## 5. 失败案例与修正：大额 Approval 未绑定参数

如果审批票据只写“允许退款”，Agent 可把 80 元审批重放到 1200 元。修正票据绑定资源和金额，资源服务器精确比较。


In [5]:
high_value11 = dict(next(request11 for request11 in requests11 if request11["id"] == "r5"))  # 复制大额退款请求。
high_value11["jti"] = "j7"  # 使用新的 proof 标识避免与原失败请求混淆。
high_value11["approval"] = "approve:refund"  # 构造未绑定资源和金额的宽泛票据。
broad_allowed11, broad_reason11 = authorize11(high_value11)  # 检查宽泛票据仍应失败。
high_value11["jti"] = "j8"  # 为正确审批使用新的 jti。
high_value11["approval"] = "approve:o-3:1200"  # 构造精确绑定资源和金额的票据。
fixed_allowed11, fixed_reason11 = authorize11(high_value11)  # 检查参数绑定审批通过。
print("失败行为：宽泛审批", broad_allowed11, broad_reason11)  # 展示未绑定动作参数的票据被拒绝。
print("修正行为：精确审批", fixed_allowed11, fixed_reason11, high_value11["approval"])  # 展示 step-up 成功路径。


失败行为：宽泛审批 False step_up_required
修正行为：精确审批 True allowed approve:o-3:1200


## 6. 生产边界与授权制品

真实 OAuth 要验证 issuer、签名、exp、nbf、client、resource indicator 和 token exchange policy。DPoP 还绑定 HTTP method/URI；replay cache 需多实例一致。审批票据应验签并有短 TTL。


In [6]:
oauth_contract11 = {"issuer": "auth.example", "audience": "orders-api", "token_ttl_seconds": 300, "proof": "DPoP", "replay": "shared_jti_cache", "object_acl": "resource_server", "step_up_threshold": 500}  # 定义委托授权发布合同。
print("Agent OAuth 制品", oauth_contract11)  # 展示 issuer、audience、proof 和审批策略。
print("生产替换点：JWT验签、token exchange、DPoP htm/htu、分布式replay cache、对象ACL和审批验签")  # 说明内存策略的边界。


Agent OAuth 制品 {'issuer': 'auth.example', 'audience': 'orders-api', 'token_ttl_seconds': 300, 'proof': 'DPoP', 'replay': 'shared_jti_cache', 'object_acl': 'resource_server', 'step_up_threshold': 500}
生产替换点：JWT验签、token exchange、DPoP htm/htu、分布式replay cache、对象ACL和审批验签


## 7. 最小回归测试

断言保护 confused deputy、对象 ACL、replay 和参数绑定审批。


In [7]:
assert len(requests11) >= 5  # 保证授权案例覆盖多类攻击与正常动作。
assert dict((row11[0], row11[2]) for row11 in audit11)["r2"] == "audience_mismatch"  # 保证错误 audience 被拒绝。
assert dict((row11[0], row11[2]) for row11 in audit11)["r3"] == "resource_forbidden"  # 保证跨租户对象被拒绝。
assert dict((row11[0], row11[2]) for row11 in audit11)["r6"] == "replay_detected"  # 保证重复 jti 被识别。
assert broad_allowed11 is False and fixed_allowed11 is True  # 保证审批必须绑定资源和金额。
print("最小回归测试通过：Audience、Scope、ACL、DPoP和Step-up稳定")  # 显示委托授权关键性质已验证。


最小回归测试通过：Audience、Scope、ACL、DPoP和Step-up稳定
